In [1]:
import sys
import os
import numpy as np

# Coordinate Transformations

WCSim uses a different coordinate system to the WCTE conventions (documented on the WCTE wiki). All analyses should use the WCTE conventions except where explicitally necessary to work with different coordinates, when working with different coordinates variable names should make it immediately clear that this is what is happening (e.g. track_vertex_wcsim_coord). The following class is designed to handle the conversion between the two as a standard library. Where you need to use please use this standard library for the conversion rather than defining something yourself.

In [2]:
from analysis_tools import WCSimCoordinateTransform

In [3]:
# Create instance
coord_transform = WCSimCoordinateTransform()

# Define your set of WCSim coordinates and specify your unit - the default is mm
#you can specify either an array of coordinates or a single set of coordinates
wcsim_coords = [0,0,0] # The center of the tank in WCSim
wcte_coords = coord_transform.wcsim_to_wcte(wcsim_coords,unit="mm")
print("Single set of coordinates",wcsim_coords,"->",wcte_coords)

wcsim_coords = [[0,0,0],[10,20,30],[1,2,50]]
wcte_coords = coord_transform.wcsim_to_wcte(wcsim_coords, unit="cm")
print("Array of coordinates",wcsim_coords,"->",wcte_coords)

#if you don't specify a unit it defaults to mm
wcsim_coords = [0,0,0]
wcte_coords = coord_transform.wcsim_to_wcte(wcsim_coords)
print("Array of coordinates",wcsim_coords,"->",wcte_coords)


Single set of coordinates [0, 0, 0] -> [  0.     424.7625   0.    ]
Array of coordinates [[0, 0, 0], [10, 20, 30], [1, 2, 50]] -> [[ 0.      42.47625  0.     ]
 [10.      62.47625 30.     ]
 [ 1.      44.47625 50.     ]]
Array of coordinates [0, 0, 0] -> [  0.     424.7625   0.    ]


In [4]:
#or do the reverse 
wcte_coords = [0,424.7625,0] # The center of the tank in WCTE coordinates
wcsim_coords = coord_transform.wcte_to_wcsim(wcte_coords,unit="mm")
print(wcsim_coords)

[0. 0. 0.]


# Mapping WCSim Tube Number to WCTE mPMT and PMT numbering convention

WCTE has a convention for how mPMTs and PMTs are numbered. This is sometimes called the slot, position of the PMT. The slot specifies the mPMT numbering in the detector while the position is the position number of the PMT in the mPMT. Read WCTE conventions for more. WCSim uses a tube_no to designate each PMT in the detector. This number depends on how many mPMTs are built in the detector (e.g. the mapping is different if you run without the empty mPMT slots). WCSim generates a txt file which can be used to generate the mapping. This mapping class loads the mapping for WCSim v1.12.29, this is the nominal configuration as of June 2026, any WCSim file can be used to generate the mapping, for debugging you can load the mapping file made when you ran the simulation for comparison.

In [5]:
from analysis_tools import WCSimPMTMapping

In [6]:
# Create instance
wcsimPMTMapping = WCSimPMTMapping() #loads the default mapping for WCSim v1.12.29 
#or specify location of mapping file with geo_file 
# wcsimPMTMapping = WCSimPMTMapping("your_geofile")

Loading default WCSim mapping for WCSim 1.12.29


if needed you can run this line to debug the default mapping with the version in your simulation runs

In [7]:
#debugging function check that the default mapping is the same as yours
wcsimPMTMapping.check_mapping_consistency_with_default("/afs/cern.ch/user/l/lcook/analysis_tools/analysis_tools/data/wcsim_v1_12_29_mapping_geofile.txt")

The specified geofile is consistent with the default geofile


Do the mapping from WCSim to WCTE convention or do the reverse mapping using the functions below. Exceptions will be raised if invalid tube or slot/pmt ids are used

In [8]:
wcsim_tube_no = [636,637,864,342]
wcte_mPMT_slots, wcte_pmt_pos  = wcsimPMTMapping.map_wcsim_tube_no_to_wcte_slot_pos(wcsim_tube_no)
print("Map to WCTE slot and positions",wcte_mPMT_slots,wcte_pmt_pos)

wcsim_tube_no = wcsimPMTMapping.map_wcte_slot_pos_to_wcsim_tube_no(wcte_mPMT_slots,wcte_pmt_pos)
print("Map back to the tube number",wcsim_tube_no)


Map to WCTE slot and positions [23 23  0 55] [ 8  9  8 18]
Map back to the tube number [636 637 864 342]


### Important note if you are using watchmal data_tools to make an npz file from WCSim output
Watchmal data tools takes the WCSim tube number and subtracts 1 from it when it makes the npz file. You will need to account for this in the mapping if you are using data dumped using that function. https://github.com/WatChMaL/DataTools/blob/ae0c4da3120a357bb9c33d88c30e4c99724006ea/root_utils/root_file_utils.py#L143(https://github.com/WatChMaL/DataTools/blob/ae0c4da3120a357bb9c33d88c30e4c99724006ea/root_utils/root_file_utils.py#L143). To simplify this a togger is added to the mappings that should be used if you are using the watchmal NPZ files 

In [9]:
wcsim_tube_no = [635,636,863,341]
wcte_mPMT_slots, wcte_pmt_pos  = wcsimPMTMapping.map_wcsim_tube_no_to_wcte_slot_pos(wcsim_tube_no, use_watchmal_npz = True)
print("Map to WCTE slot and positions",wcte_mPMT_slots,wcte_pmt_pos)

wcsim_tube_no = wcsimPMTMapping.map_wcte_slot_pos_to_wcsim_tube_no(wcte_mPMT_slots,wcte_pmt_pos, use_watchmal_npz = True)
print("Map back to the tube number in watchmal convention",wcsim_tube_no)


Map to WCTE slot and positions [23 23  0 55] [ 8  9  8 18]
Map back to the tube number in watchmal convention [635 636 863 341]
